# 23 — Choosing: Plain API Management vs. AI Gateway Policies

Notebooks 16-22 showed you *how* to expose, secure, govern, and call MCP
servers and LLM backends through Azure API Management. This notebook
answers a different question: **when should you reach for an AI Gateway
policy versus a plain, ordinary API Management policy you'd use for any
REST API?**

This matters because "just enable AI Gateway" isn't a real decision --
AI Gateway isn't a switch, it's a set of specific policies
(`llm-token-limit`, `llm-semantic-cache-lookup`/`-store`,
`llm-content-safety`, and friends) layered on the API gateway you already
have. Every API Management instance already *has* an API gateway; the
question is only which policies a given API actually needs.

Read alongside [docs/11_apim_vs_ai_gateway_decision_guide.md](../docs/11_apim_vs_ai_gateway_decision_guide.md)
and try the recipes in
[examples/apim_policies/](../examples/apim_policies/README.md).

## The core distinction

| | Plain API Management policies | AI Gateway policies |
| --- | --- | --- |
| Governs by | Call count, bandwidth, concurrency | LLM **tokens**, semantic **similarity**, content **harm categories** |
| Examples | `rate-limit-by-key`, `quota-by-key`, `cache-lookup`/`cache-store`, `ip-filter`, `validate-azure-ad-token` | `llm-token-limit`, `llm-emit-token-metric`, `llm-semantic-cache-lookup`/`-store`, `llm-content-safety` |
| Applies to | Any REST API, including MCP servers' tool-call traffic | LLM chat/completions/embeddings APIs (OpenAI, Azure OpenAI, Anthropic, Vertex AI schemas) |
| Extra prerequisites | None beyond API Management itself | Often needs an embeddings deployment, a Content Safety resource, or Application Insights wired up |

Both families are official, first-class Azure API Management policies --
"AI Gateway" is Microsoft's name for the *category*, in the same policy
reference page as `rate-limit` and `cache-lookup`
([api-management-policies](https://learn.microsoft.com/en-us/azure/api-management/api-management-policies#ai-gateway)).

## Decision table: signal → policy family

| If you're seeing... | Reach for... | Family |
| --- | --- | --- |
| A client hammering an endpoint with request volume | `rate-limit-by-key` / `quota-by-key` | Plain |
| One request silently costing 50x another (long prompts) | `llm-token-limit` / `azure-openai-token-limit` | AI Gateway |
| Repeated identical requests wasting backend capacity | `cache-lookup` / `cache-store` | Plain |
| Repeated *differently worded* requests with the same meaning | `llm-semantic-cache-lookup` / `-store` | AI Gateway |
| No visibility into who's driving cost | `emit-metric` (generic) | Plain |
| No visibility into who's driving cost, and cost = tokens | `llm-emit-token-metric` | AI Gateway |
| Untrusted users submitting free-text prompts | `llm-content-safety` | AI Gateway |
| A single backend that's a single point of failure | backend pool + circuit breaker | Both -- resiliency isn't AI-specific, but LLM backends benefit the most (notebook 21) |
| Callers need to be authenticated | `validate-azure-ad-token` / `validate-jwt` | Plain -- identical whether the backend is an LLM or not |

## A five-question decision tree

1. **Does the backend consume or produce natural-language tokens** (a chat/completions
   API, an embeddings API, or an MCP tool whose result feeds an LLM)?
   - **No** → you almost certainly only need plain API Management policies.
     Stop here.
   - **Yes** → continue.
2. **Do you need per-caller cost or quota control finer than "N calls"?**
   → `llm-token-limit` / `azure-openai-token-limit`
   ([examples/apim_policies/ai_gateway/01](../examples/apim_policies/ai_gateway/01_llm_token_limit.xml)).
3. **Do near-duplicate prompts show up often enough that caching by meaning
   would help?** → `llm-semantic-cache-lookup` / `-store`
   ([ai_gateway/06](../examples/apim_policies/ai_gateway/06_llm_semantic_cache_lookup_and_store.xml)).
4. **Could a prompt or response contain harmful content you must block?**
   → `llm-content-safety`
   ([ai_gateway/09](../examples/apim_policies/ai_gateway/09_llm_content_safety.xml)).
5. **Do you have (or want) more than one model deployment?** → a backend
   pool with load balancing and a circuit breaker
   ([ai_gateway/11-15](../examples/apim_policies/ai_gateway/)).

Answering "no" to 2-5 doesn't mean you made a mistake -- it means AI
Gateway's *extra* capabilities don't apply to your scenario yet, and plain
API Management policies (auth, plain rate limiting, tracing) are still
exactly what you should use.